# OpenHeadMouse: Session viewer (Jupyter Notebook Version)
There is a version of the session viewer in the runtime executable. This version is designed for academic analysis of head movement/mouse data. For more information, see the project on GitHub: https://github.com/simonjudge/OpenHeadMouse 
To make this work: run (press the play button for each section) the setup, loading and session-viewer cells in order (waiting for them to complete)

**Privacy:** recordings uploaded here go to your Google Colab runtime, not to GitHub or any playback service. 
Do not share a notebook with saved outputs containing participant data. Runtime files and imports are temporary; download the summary CSV before disconnecting.

In [1]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/simonjudge/OpenHeadMouse.git'
REVISION = ''  # Optional published branch or tag; empty uses the default branch.
# Prefer this notebook's local project over an older nested clone.
PROJECT_DIR = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                    if (folder / 'colab_session_viewer' / 'app.py').is_file()
                    and (folder / 'core').is_dir()), Path.cwd() / 'OpenHeadMouse')

if not PROJECT_DIR.exists():
    command = ['git', 'clone', '--depth', '1']
    if REVISION:
        command += ['--branch', REVISION]
    subprocess.check_call(command + [REPOSITORY_URL, str(PROJECT_DIR)])
requirements = PROJECT_DIR / 'colab_session_viewer' / 'requirements.txt'
if not requirements.is_file():
    raise FileNotFoundError('This checkout does not contain the Colab viewer. Select its published branch or upload the complete changed project.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])
PROJECT_DIR = PROJECT_DIR.resolve()
sys.path = [str(PROJECT_DIR)] + [entry for entry in sys.path if entry != str(PROJECT_DIR)]
loaded_viewer = sys.modules.get('colab_session_viewer')
if loaded_viewer is not None and Path(loaded_viewer.__file__).resolve().parent != PROJECT_DIR / 'colab_session_viewer':
    raise RuntimeError('Another viewer checkout is already loaded. Restart the kernel and rerun setup.')

from session_viewer.mesh_visualizer import download_mediapipe_model
download_mediapipe_model()
print('Ready. Run the loading cell, then the session-viewer cell below.')


Ready. Run the loading cell, then the session-viewer cell below.


In [2]:
try:
    from google.colab import output
except ModuleNotFoundError as error:
    if error.name not in ('google', 'google.colab'):
        raise
    print('Local notebook runtime; Colab widget manager not needed.')
else:
    output.enable_custom_widget_manager()
import importlib
import colab_session_viewer.dwell as viewer_dwell
import colab_session_viewer.data as viewer_data
import colab_session_viewer.archive as viewer_archive
import colab_session_viewer.cursor_paths as viewer_cursor_paths
import colab_session_viewer.selection_stats as viewer_selection_stats
import colab_session_viewer.outliers as viewer_outliers
import colab_session_viewer.app as viewer_app
for module in (viewer_dwell, viewer_data, viewer_archive, viewer_cursor_paths, viewer_selection_stats, viewer_outliers, viewer_app):
    importlib.reload(module)
from colab_session_viewer.app import NotebookViewer
print('Viewer loaded from:', viewer_app.__file__)

DEMO_DATA_DIR = str(PROJECT_DIR / 'colab_session_viewer' / 'demodata')  # Bundled demo ZIPs; works locally and in Colab.
DEMO_MODE = False  # True: automatically load demos and show only 'Loaded Demo Data' instead of import controls.
if 'viewer' not in globals():
    viewer = NotebookViewer(demo_directory=DEMO_DATA_DIR, demo_mode=DEMO_MODE)
viewer.show_loading();


Local notebook runtime; Colab widget manager not needed.
Viewer loaded from: C:\Users\judges.BDGH\Documents\VSCode\HeadMouse\colab_session_viewer\app.py


## Session Viewer: Data and Analysis
Run this cell after the loading cell. New imports update the viewer automatically, and rerunning this cell keeps your loaded sessions.

**Full screen in Colab:** After running the cell, select it and open the **Command palette**. Search for **View output fullscreen** and run that command. Running a notebook cell cannot reliably enter browser full screen on its own. In JupyterLab, right-click the output and choose **Create New View for Output** to open the viewer in its own tab.


In [ ]:
viewer.show_session();


## Notes
### Load and label recordings

- **Try the demos:** Tick **Load Demo Data**. The bundled ZIPs work in Colab and locally without Google Drive. To use a different demo folder, change **Demo folder** or `DEMO_DATA_DIR`. Unticking does not unload the demos; restart the kernel to reload a ZIP that changed at the same path.
- **Add one session:** Enter a participant (e.g. `P01`) and session (e.g. `Test 1`), then select a head recording (JSON/CSV), a Fitts pointing test (JSON), or both. Click **Add session** and repeat for each test. Pair files only if they were recorded together and their timestamps overlap.
- **Import many sessions:** Use **Dataset ZIP → Import dataset ZIP**. A ZIP can include a manifest: a CSV that tells the viewer which files belong to each participant and session. Without one, the viewer suggests head/Fitts pairs using overlapping timestamps from a shared clock. It does not guess participant names, and it leaves uncertain pairs separate.

For a ZIP without a manifest, review the suggested pairs and edit the participant and session names. Use **Add assignment row** to split an incorrect pair, or untick a row to skip it. Select **Import labelled sessions**, then **Download manifest.csv for ZIP import N** to save your choices. Put that CSV at the top level of the original ZIP to reuse the labels and pairings. Keep the ZIP: the manifest stores assignments, not recordings or analysis settings.

A ZIP can contain one valid manifest, with any CSV filename at the top level. Only files listed in the manifest are imported. Multiple ZIPs can be added to the same collection, but duplicate participant/session pairs are rejected, not overwritten. Use `/` in paths relative to the ZIP and leave the head or Fitts path blank if that recording was not made. The first four columns below are required; the last two are optional.
```csv
participant,session,head_file,fitts_file,selection_method,dwell_time_s
P01,Test 1,P01/head1.json,P01/fitts1.json,external_switch,
P01,Test 2,P01/head2.csv,P01/fitts2.json,external_switch,
P02,Test 1,,P02/fitts1.json,dwell,0.6
```
Selection methods are `external_switch`, `dwell`, `mouse`, `other` (non-dwell), and `unknown`. Use `dwell` when holding the cursor in place selects a target, and enter the configured hold time in seconds. Leave the time blank for other methods. The viewer does not guess an unlabelled selection method.

### Watch a session and inspect cursor paths

- Choose a **Participant** and **Session** above playback. Play, rewind, or drag the timeline to inspect a moment. You can change speed, zoom the graphs, and rotate the head view.
- **Misses** appear as red targets in playback and pink bands on the cursor graphs. Colour shows the saved outcome for the whole target interval, even before the click. Excluded periods are generally grey; missed targets keep their miss colours in the cursor display. **Current target planned ID** is a difficulty score, not the target number.
- **Cumulative cursor paths** overlays one participant's paths across their tests. Two 12-target tests are numbered 1-24; exclusions do not renumber the targets. Type `1-12` or `1,2,24` in **Targets**, use **Next** / **Previous** to step through paths, or choose **All**. If no path was recorded, only the click is shown. Use **Download image (SVG)** below the plot to save the current selection.

### Choose what to leave out of the results

| Control | What it changes |
| --- | --- |
| **First targets per Fitts file** | Skips the first N completed targets in **every file**, not whole tests. Also omits the corresponding time intervals from head calculations. Paired recordings need timestamps from a shared clock to do this. |
| **Skip opening of head recording (s)** | Skips the opening of each head recording for whole-recording movement estimates only (range, speed and acceleration). It does not change initial pose or the movement exercise. |
| **Each hold opening (s)** | Lets the head settle by skipping the start of **each** still hold, including initial rest and each direction in the movement exercise. It affects head-pose, exercise and movement estimates. Initial rest uses only the labelled hold, not the move into position. |

Both time controls default to **1 second**; enter **0** to keep the full recording or hold for that control. Neither affects pointing results. Source files, calibration and playable data stay intact. If nothing remains for a calculation, its result is blank. Original recording counts and durations are not reduced.

**Target review** in **Fitts results** flags unusual trials for inspection, not automatic deletion. It looks for movement time, start-to-click distance, or how far the click landed short of or beyond the target centre along the line of movement, more than three standard deviations from that participant's average. A standard deviation describes the usual spread of values. The comparison uses the participant's remaining trials after the first-target rule and does not change as you untick items. Untick **Include** to remove a trial from the three Fitts results tables and their exports; **Include all reviewed targets** reverses those decisions. Review does not change playback, cursor paths or head statistics.

### Read the tables and downloads

Each main table has one row per participant. Hover over a column heading for a short explanation, or open **Table Notes → Field key** for definitions and calculations. **Table Notes → Individual sessions** shows each test's results and source filenames. **Contributing targets and omissions**, where shown, explains which trials were used. Download a CSV to work with the underlying values; for example, the screen shows an error rate as **8.3% (2/24)**, while the CSV stores the unrounded fraction.

| Pointing results | How to read them |
| --- | --- |
| **Fitts targets and hits** | Counts and saved target settings for the analysed trials, plus the range of planned difficulty. Wider or closer targets are generally easier. For circle tests, saved amplitude is the circle radius, not necessarily the distance the cursor moved. |
| **Fitts stats** | Uses the recorded selections. **Planned ID** comes from target distance and width; **Effective ID** also reflects where clicks actually landed. See **Fitts calculation review** for limits on comparing these pooled results with other studies. |
| **No-Dwell stats** | Estimates movement time without the configured dwell wait. Trials with an unknown selection method or impossible adjusted times are left out and counted in the notes. |
| **First-entry selection stats** | Estimates how long it took the cursor to first reach the target. Trials with no recorded path or no detected contact are left out. A 0% error rate here does not mean every recorded attempt succeeded. See [Ching, MacKenzie and Allison (2025)](https://doi.org/10.1145/3694907.3765916). |

For the saved settings and observed trial counts by file, open **Fitts stats → Table Notes → Fitts test conditions by file**. To see the optional dwell estimates, enable **Show dwell behaviour estimates** under **Fitts calculation review**.

### Understand the head plots

- **Yaw** is turning left or right, **pitch** is looking up or down, and **roll** is tilting an ear toward a shoulder. The zero point depends on the tracker and calibration; recorded yaw is not necessarily a compass heading. See **Angle references and zero-position diagrams** in the initial-position table notes for sensor-specific details.
- **Range of movement stats** describes the labelled exercise. **Head movement stats** covers the wider recording, which can include preparation and the exercise; the reported range for each angle is the largest within a single recording, not the span across recordings.
- **Zero yaw / Zero pitch / Zero roll** redraws the participant pose plots only. It does not change playback, tables or CSVs. **Download SVG** beside a participant's heading saves the plot as currently shown. Treat downloaded plots as participant data.

### Load from Google Drive (optional)

For recordings already on Google Drive, mount Drive in a separate cell and call:
```python
viewer.add_session('P03', 'Test 1',
    head_path='/content/drive/MyDrive/study/head.json',
    fitts_path='/content/drive/MyDrive/study/fitts.json')
```